In [ ]:
from pathlib import Path
from langchain_community.document_loaders import (
    TextLoader,
)
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/vc/qs2_0j9d3h98ms0t6h_ls2xc0000gp/T/ipykernel_92222/1632223892.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/Users/user2/Documents/Github_Repos/chatbot_Luat_nghi_dinh/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
md_dir = Path("../data/md")
all_documents = []
for path in md_dir.glob("*.md"):
    print(path.name.split(".")[0])

362-2026-NĐ-CP
54-2026-TT-BCT
09-2026-TT-KTNN
53-2026-TT-BCT
06-2026-TT-BDTTG
52-2026-TT-BCT
358-2026-NĐ-CP
55-2026-TT-BCT
20-2026-TT-BNV
57-2026-TT-BCT
136-2026-TT-BTC
07-2026-TT-BDTTG
58-2026-TT-BCT
59-2026-TT-BCT
51-2026-TT-BCT
138-2026-TT-BTC
45-2026-QĐ-TTg
137-2026-TT-BTC
56-2026-TT-BCT
368-2026-NĐ-CP
47-2026-TT-NHNN
69-2026-TT-BXD
357-2026-NĐ-CP
71-2026-TT-BXD
359-2026-NĐ-CP
70-2026-TT-BXD
60-2026-TT-BCT


In [4]:
import re

MARKER_RE = re.compile(r"^[A-Z][A-Z_\-]+$")
WANTED = {
    "DOCUMENT_TYPE": "document_type",
    "DOCUMENT_TITLE": "document_title",
    "PREAMBLE": "preamble",
}

#@ Lấy loại, tiêu đề và phần căn cứ từ các marker của văn bản Markdown.
def extract_sections(text: str) -> dict[str, str]:
    sections : dict[str, list[str]]= {}
    current : str|None= None  #? Khóa metadata của section đang mở.

    for raw in text.splitlines():
        line = raw.strip()

        #? Marker mới thay section đang mở; chỉ giữ các loại trong WANTED.
        if MARKER_RE.match(line):
            current = WANTED.get(line)
            if current:
                sections[current] = []
            continue

        #? Chỉ thu thập dòng có nội dung trong section cần lấy.
        if current and line:
            sections[current].append(line)

    return {
        "document_type": " ".join(sections.get("document_type", [])),
        #? Tiêu đề bị ngắt dòng giữa câu nên nối bằng khoảng trắng.
        "document_title": " ".join(sections.get("document_title", [])),
        #? Giữ xuống dòng giữa các căn cứ trong phần mở đầu.
        "preamble": "\n".join(sections.get("preamble", [])),
    }

In [5]:
#@ Đọc các file Markdown và tạo Document kèm metadata từ YAML đầu file.
def turn_files_into_documents(data_path:Path = Path("../data/md")) -> list[Document]:
    all_documents :list[Document] = []
    for path in data_path.glob("*.md"):
        with open(path, "r", encoding="utf-8") as f:
            f.readline()
            status = f.readline().split(":", 1)[1].strip()
            issue_date = f.readline().split(":", 1)[1].strip().strip("'")
            eff_from = f.readline().split(":", 1)[1].strip().strip("'")
            eff_to = f.readline().split(":", 1)[1].strip().strip("'")

        loader = TextLoader(file_path=path, encoding="utf-8")
        doc : Document = loader.load()[0]

        doc.metadata.update({
            "name": path.stem,
            "status": status,
            "issue_date": issue_date,
            "eff_from": eff_from,
            "eff_to": eff_to,
            **extract_sections(doc.page_content),
        })
        all_documents.append(doc)
    return all_documents

In [6]:
all_documents : list[Document] = turn_files_into_documents()

In [7]:
from dataclasses import dataclass, field
from langchain_core.documents import Document


@dataclass(frozen=True)
class Level:  #? Cấu hình marker và khóa metadata cho một tầng tách văn bản.
    boundaries: frozenset[str]            #? Các marker dùng để cắt ở tầng này.
    boundary_key: str                     #? Khóa metadata lấy từ dòng có nội dung sau marker.
    extra: dict[str, str] = field(default_factory=lambda: {})  #? Ánh xạ khóa metadata sang marker bổ sung.


LEVELS: list[Level] = [
    Level(
        boundaries=frozenset({"PROV_CHAPTER", "PROV_APPENDIX"}),
        boundary_key="chapter",
        extra={"chapter_title": "PROV_CHAPTER_TITLE"},
    ),
    Level(
        boundaries=frozenset({"PROV-ARTICLE"}),
        boundary_key="article",
    ),
]


#@ Lấy dòng có nội dung đầu tiên sau marker, hoặc từ đầu nếu không có marker.
def _line_below(lines: list[str], marker: str | None = None) -> str | None:
    """Dòng non-empty đầu tiên ngay dưới `marker`. marker=None thì lấy từ đầu."""
    start = 0
    if marker is not None:
        for i, raw in enumerate(lines):
            if raw.strip() == marker:
                start = i + 1
                break
        else:
            return None
    for raw in lines[start:]:
        if raw.strip():
            return raw.strip()
    return None


#@ Cắt văn bản theo các marker ranh giới thành cặp marker và nội dung.
def _cut(text: str, boundaries: frozenset[str]) -> list[tuple[str, str]]:
    """Cắt text thành [(marker, content)], bỏ phần trước boundary đầu tiên."""
    parts: list[tuple[str, str]] = []
    kind: str | None = None
    buf: list[str] = []

    for raw in text.splitlines():
        if raw.strip() in boundaries:
            if kind is not None:
                parts.append((kind, "\n".join(buf).strip()))
            kind, buf = raw.strip(), []
        elif kind is not None:
            buf.append(raw.rstrip())

    if kind is not None:
        parts.append((kind, "\n".join(buf).strip()))
    return parts


#@ Tách Document lần lượt theo các tầng và chuyển thông tin section vào metadata.
def split_document(documents: list[Document], levels: list[Level] = LEVELS) -> list[Document]:
    #? Hết tầng thì trả lại danh sách hiện có.
    if not levels:
        return documents

    level, rest = levels[0], levels[1:]
    next_docs: list[Document] = []

    for d in documents:
        parts = _cut(d.page_content, level.boundaries)

        #? Giữ nguyên Document không có boundary để tầng sau tiếp tục xử lý.
        if not parts:
            next_docs.append(d)
            continue

        for kind, content in parts:
            lines = content.splitlines()
            meta = {**d.metadata, "section_kind": kind}

            if (v := _line_below(lines)) is not None:
                meta[level.boundary_key] = v
            for key, marker in level.extra.items():
                if (v := _line_below(lines, marker)) is not None:
                    meta[key] = v

            next_docs.append(Document(page_content=content, metadata=meta))

    #? Đưa toàn bộ kết quả của tầng này qua tầng kế tiếp.
    return split_document(next_docs, rest)

In [8]:
sections:list[Document] = split_document(all_documents)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    separators = ["\n\n", "\n", " ", ""],
    length_function = len,
)
chunks : list[Document] = splitter.split_documents(sections)
print("Số chunk:", len(chunks))
print("Metadata chunk đầu:", chunks[0].metadata)

Số chunk: 2007
Metadata chunk đầu: {'source': '../data/md/362-2026-NĐ-CP.md', 'name': '362-2026-NĐ-CP', 'status': 'Chưa có hiệu lực', 'issue_date': '2026-09-18', 'eff_from': '2026-11-02', 'eff_to': 'null', 'document_type': 'NGHỊ ĐỊNH', 'document_title': 'Sửa đổi, bổ sung một số điều của Nghị định số 95/2016/NĐ-CP ngày 01 tháng 7 năm 2016 của Chính phủ quy định về xử phạt vi phạm hành chính trong lĩnh vực thống kê được sửa đổi, bổ sung bởi Nghị định số 100/2021/NĐ-CP ngày 15 tháng 11 năm 2021 của Chính phủ', 'preamble': 'Căn cứ Luật Tổ chức Chính phủ số 63/2025/QH15;\nCăn cứ Luật Tổ chức chính quyền địa phương số 72/2025/QH15;\nCăn cứ Luật Xử lý vi phạm hành chính số 15/2012/QH13 được sửa đổi, bổ sung bởi Luật số 67/2020/QH14 và Luật số 88/2025/QH15;\nCăn cứ Luật Thống kê số 89/2015/QH13 được sửa đổi, bổ sung bởi Luật số 01/2021/QH15 và Luật số 138/2025/QH15;\nTheo đề nghị của Bộ trưởng Bộ Tài chính;\nChính phủ ban hành Nghị định sửa đổi, bổ sung một số điều của Nghị định số 95/2016/NĐ-

In [10]:
device = "mps"

In [12]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:  #? Quản lý model SentenceTransformer và tạo embedding cho văn bản.
    #@ Lưu cấu hình và nạp model embedding khi khởi tạo.
    def __init__(self, model_name: str = "AITeamVN/Vietnamese_Embedding", device:str = "cpu"):
        self.model_name = model_name
        self.device = device
        self.model: SentenceTransformer = self._load_model()

    #@ Nạp model theo tên đã cấu hình và báo lỗi nếu nạp thất bại.
    def _load_model(self):
        try:
            return SentenceTransformer(self.model_name)
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    #@ Mã hóa tập văn bản đầu vào và trả về các vector embedding.
    def generate_embeddings(self, texts):
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager(device = device)
embedding_manager

KeyboardInterrupt: 

In [ ]:
embedding_manager.generate_embeddings("xin chào tôi là người Việt Nam")